# 서울시 상권분석 EDA

한국어 컬럼으로 변환된 데이터를 불러옵니다.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)

## 한국어 데이터 불러오기

In [ ]:
DATA_DIR = Path("../data/korean")

DATA_NAMES = [
    "trade_area",
    "stores",
    "sales",
    "floating_population",
    "resident_population",
    "working_population",
    "facilities",
]

datasets = {
    name: pd.read_csv(DATA_DIR / f"{name}.csv")
    for name in DATA_NAMES
}

datasets.keys()

In [ ]:
datasets["trade_area"].head()

## 1. 데이터셋 크기 확인

각 데이터의 행 수와 열 수를 확인한다. 행 수가 0이거나 예상한 컬럼 수와 다르면 수집 범위와 저장 과정의 확인이 필요하다. 데이터마다 분석 단위가 다르므로 행 수가 서로 같을 필요는 없다.

In [30]:
dataset_sizes = pd.DataFrame([
    {"데이터": name, "행 수": df.shape[0], "열 수": df.shape[1]}
    for name, df in datasets.items()
])

dataset_sizes

,데이터,행 수,열 수
0,trade_area,1650,15
1,stores,1604844,18
2,sales,460329,59
3,floating_population,34633,31
4,resident_population,34275,33
5,working_population,34386,30
6,facilities,33138,29


## 2. 결측값 확인

결측값이 있는 데이터와 컬럼, 개수, 비율을 확인한다. 핵심 키의 결측은 데이터 식별이 불가능하므로 문제로 판단한다. 일반 지표와 관리 컬럼의 결측은 컬럼의 의미와 수집 방식을 확인한 뒤 판단한다.

In [28]:
missing_list = []

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    for column, count in missing.items():
        missing_rate = count / len(df) * 100
        missing_list.append([name, column, count, missing_rate])

missing_df = pd.DataFrame(
    missing_list,
    columns=["데이터", "컬럼", "결측값 수", "결측률 (%)"],
)

missing_df["결측률 (%)"] = missing_df["결측률 (%)"].round(2)

missing_df

,데이터,컬럼,결측값 수,결측률 (%)
0,stores,원본_기준_일자,1604844,100.00
1,sales,원본_기준_일자,460329,100.00
2,floating_population,원본_기준_일자,34633,100.00
3,resident_population,원본_기준_일자,34275,100.00
4,working_population,원본_기준_일자,34386,100.00
5,facilities,원본_기준_일자,33138,100.00


## 3. 중복 행 확인

모든 컬럼 값이 완전히 동일한 행을 확인한다. `중복 행 수`가 0이면 정상이다. 0보다 크면 같은 데이터가 반복 저장된 것인지 확인하되, 첫 번째 행은 원본으로 보고 이후 반복된 행만 중복으로 계산한다.

In [ ]:
duplicate_list = []

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_rate = duplicate_count / len(df) * 100

    duplicate_list.append([name, duplicate_count, duplicate_rate])

duplicate_df = pd.DataFrame(
    duplicate_list,
    columns=["데이터", "중복 행 수", "중복률 (%)"],
)

duplicate_df["중복률 (%)"] = duplicate_df["중복률 (%)"].round(2)

duplicate_df

## 4. 데이터 유효성 탐색

통계적으로 드문 값을 임의로 제거하지 않고, 데이터 구조와 의미가 명확한 최소 조건을 검사한다.

### 4.1 수치형 기술통계

숫자로 읽힌 코드 컬럼을 제외하고 수치형 지표의 분포를 확인한다. `min`과 `max`로 범위를 보고, `mean`과 `50%`의 차이로 치우침을 살펴본다. 이 표만으로 이상값을 확정하지는 않는다.

In [ ]:
for name, df in datasets.items():
    numeric_columns = [
        column
        for column in df.select_dtypes(include="number").columns
        if "코드" not in column
    ]

    print(f"[{name}]")
    display(
        df[numeric_columns]
        .describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
        .T
    )

### 4.2 핵심 키 고유성

각 데이터의 분석 단위를 나타내는 핵심 키 조합이 한 행씩만 존재하는지 확인한다. `전체 행 수`와 `고유 키 수`가 같고 `중복 키 수`가 0이면 정상이다.

In [ ]:
KEY_COLUMNS = {
    "trade_area": ["상권_코드"],
    "stores": ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
    "sales": ["기준_년분기_코드", "상권_코드", "서비스_업종_코드"],
    "floating_population": ["기준_년분기_코드", "상권_코드"],
    "resident_population": ["기준_년분기_코드", "상권_코드"],
    "working_population": ["기준_년분기_코드", "상권_코드"],
    "facilities": ["기준_년분기_코드", "상권_코드"],
}

key_check_list = []

for name, key_columns in KEY_COLUMNS.items():
    df = datasets[name]
    unique_key_count = df[key_columns].drop_duplicates().shape[0]
    duplicate_key_count = df.duplicated(subset=key_columns).sum()

    key_check_list.append([
        name,
        " + ".join(key_columns),
        len(df),
        unique_key_count,
        duplicate_key_count,
    ])

key_check_df = pd.DataFrame(
    key_check_list,
    columns=["데이터", "핵심 키", "전체 행 수", "고유 키 수", "중복 키 수"],
)

key_check_df

### 4.3 상권 코드 참조 확인

점포·매출·인구·시설 데이터의 상권 코드가 상권 마스터인 `trade_area`에 존재하는지 확인한다. `미등록 상권 코드 수`와 `미등록 행 수`가 모두 0이면 정상이다. 데이터별 상권 코드 수 자체는 수집 기간과 제공 범위에 따라 다를 수 있다.

In [ ]:
trade_area_codes = set(datasets["trade_area"]["상권_코드"])
reference_list = []

for name, df in datasets.items():
    if name == "trade_area":
        continue

    not_in_master = ~df["상권_코드"].isin(trade_area_codes)

    reference_list.append([
        name,
        df["상권_코드"].nunique(),
        df.loc[not_in_master, "상권_코드"].nunique(),
        not_in_master.sum(),
    ])

reference_df = pd.DataFrame(
    reference_list,
    columns=["데이터", "상권 코드 수", "미등록 상권 코드 수", "미등록 행 수"],
)

reference_df

### 4.4 코드와 명칭의 일관성

하나의 코드에 서로 다른 명칭이 연결되어 있는지 확인한다. `여러 명칭이 연결된 코드 수`가 0이면 정상이다. 0보다 크면 명칭 변경, 오탈자 또는 코드 매핑 문제인지 확인한다.

In [ ]:
CODE_NAME_PAIRS = [
    ("상권_코드", "상권_코드명"),
    ("상권_구분_코드", "상권_구분_코드명"),
    ("서비스_업종_코드", "서비스_업종_코드명"),
    ("자치구_코드", "자치구_명"),
    ("행정동_코드", "행정동_명"),
]

code_name_list = []

for name, df in datasets.items():
    for code_column, name_column in CODE_NAME_PAIRS:
        if code_column in df.columns and name_column in df.columns:
            name_count = df.groupby(code_column)[name_column].nunique()
            inconsistent_count = (name_count > 1).sum()

            code_name_list.append([
                name,
                code_column,
                name_column,
                inconsistent_count,
            ])

code_name_df = pd.DataFrame(
    code_name_list,
    columns=["데이터", "코드 컬럼", "명칭 컬럼", "여러 명칭이 연결된 코드 수"],
)

code_name_df

### 4.5 수치형 값의 최소 조건

금액·인구·점포·시설 수는 `음수 개수`와 `무한대 개수`가 0이어야 한다. 건수형 컬럼은 `소수 개수`도 0이어야 한다. 비율은 0~100, 면적은 0 초과를 정상 범위로 본다. `0 개수`는 참고값이며 매출·점포·시설에서 0은 정상일 수 있으므로 오류로 판단하지 않는다.

In [ ]:
numeric_check_list = []

for name, df in datasets.items():
    numeric_columns = [
        column
        for column in df.select_dtypes(include="number").columns
        if "코드" not in column
    ]

    for column in numeric_columns:
        values = df[column]
        negative_count = (values < 0).sum()
        zero_count = (values == 0).sum()
        infinity_count = values.isin([float("inf"), float("-inf")]).sum()

        decimal_count = 0
        if column.endswith("_수") or column.endswith("_건수"):
            decimal_count = ((values.dropna() % 1) != 0).sum()

        range_error_count = 0
        if column.endswith("_율"):
            range_error_count = ((values < 0) | (values > 100)).sum()
        elif "면적" in column:
            range_error_count = (values <= 0).sum()

        numeric_check_list.append([
            name, column, values.min(), values.max(),
            negative_count, zero_count, decimal_count,
            infinity_count, range_error_count,
        ])

numeric_check_df = pd.DataFrame(
    numeric_check_list,
    columns=[
        "데이터", "컬럼", "최솟값", "최댓값", "음수 개수",
        "0 개수", "소수 개수", "무한대 개수", "범위 오류 개수",
    ],
)

numeric_check_df

### 4.6 분기 코드 형식

분기 코드가 연도 4자리와 분기 1자리로 구성됐는지 확인한다. 예를 들어 `20241`은 2024년 1분기를 의미한다. 마지막 자리는 1~4만 허용하며 `형식 오류 수`가 0이면 정상이다.

In [ ]:
quarter_list = []

for name, df in datasets.items():
    if "기준_년분기_코드" not in df.columns:
        continue

    quarter = df["기준_년분기_코드"].astype("string")
    invalid_count = (~quarter.str.fullmatch(r"\d{4}[1-4]", na=False)).sum()

    quarter_list.append([
        name,
        quarter.nunique(),
        quarter.min(),
        quarter.max(),
        invalid_count,
    ])

quarter_df = pd.DataFrame(
    quarter_list,
    columns=["데이터", "분기 수", "최초 분기", "최근 분기", "형식 오류 수"],
)

quarter_df

### 4.7 주요 총계와 구성값의 일치

총계가 주요 구성값의 합과 일치하는지 확인한다. 원천 데이터의 반올림을 고려해 절대 차이가 1 이하이면 일치로 판단한다. `불일치 행 수`가 0이면 정상이다. 세부 항목이 전체 구성을 완전히 포함한다고 확신할 수 없는 시설 데이터는 이 검사에서 제외한다.

In [ ]:
SUM_RULES = {
    "stores": [
        ("유사업종 점포 수", "유사업종_점포_수", ["일반_점포_수", "프랜차이즈_점포_수"]),
    ],
    "sales": [
        ("분기 매출 금액", "분기_매출_금액", ["주중_매출_금액", "주말_매출_금액"]),
        ("분기 매출 건수", "분기_매출_건수", ["주중_매출_건수", "주말_매출_건수"]),
    ],
    "floating_population": [
        ("총 유동인구 수", "총_유동인구_수", ["남성_유동인구_수", "여성_유동인구_수"]),
    ],
    "resident_population": [
        ("총 상주인구 수", "총_상주인구_수", ["남성_상주인구_수", "여성_상주인구_수"]),
        ("총 가구 수", "총_가구_수", ["아파트_가구_수", "비아파트_가구_수"]),
    ],
    "working_population": [
        ("총 직장인구 수", "총_직장인구_수", ["남성_직장인구_수", "여성_직장인구_수"]),
    ],
}

sum_check_list = []

for name, rules in SUM_RULES.items():
    df = datasets[name]

    for check_name, total_column, part_columns in rules:
        part_sum = df[part_columns].sum(axis=1, min_count=len(part_columns))
        difference = (df[total_column] - part_sum).abs()
        mismatch_count = (difference > 1).sum()

        sum_check_list.append([
            name, check_name, mismatch_count, difference.max(),
        ])

sum_check_df = pd.DataFrame(
    sum_check_list,
    columns=["데이터", "검사 항목", "불일치 행 수", "최대 차이"],
)

sum_check_df

## 5. 데이터 유효성 검사 결과 해석

| 항목 | 판단 | 분석 결과 |
|---|---|---|
| 4.1 수치형 기술통계 | 참고 | 수치형 지표에서 음수·무한대·소수형 건수는 발견되지 않았다. 매출·인구·시설 데이터에는 0이 다수 존재하지만, 영업이나 시설이 없는 경우를 나타낼 수 있으므로 0 자체를 오류로 판단하지 않는다. 기술통계의 큰 최댓값은 실제 대형 상권일 수 있어 별도 아웃라이어로 제거하지 않는다. |
| 4.2 핵심 키 고유성 | 정상 | 7개 데이터 모두 전체 행 수와 고유 핵심 키 조합 수가 같고 중복 키 수가 0이다. 따라서 현재 정의한 분석 단위에서 중복 레코드는 없다. |
| 4.3 상권 코드 참조 | 정상 | 모든 데이터의 상권 코드가 `trade_area`에 등록되어 있으며 미등록 상권 코드와 미등록 행은 0건이다. 데이터별 상권 수 차이는 제공 범위 차이로 볼 수 있다. |
| 4.4 코드와 명칭의 일관성 | 검토 필요 | 시계열 데이터에서 상권 코드 2개가 서로 다른 명칭을 사용한다. `3110024`는 `혜화동주민센터`와 `혜회동주민센터`, `3110379`는 `KT&G 북부지사`와 `KTNG 북부지사`가 함께 존재한다. 시기별 명칭 변경 또는 표기 차이로 보이며, 분석용 명칭은 `trade_area` 기준으로 통일할지 결정할 필요가 있다. |
| 4.5 수치형 최소 조건 | 대체로 정상 | 모든 데이터에서 음수·무한대·계수형 소수값은 0건이다. 다만 `stores`에서 개업률 100 초과가 14건(최대 200), 폐업률 100 초과가 767건(최대 500) 확인됐다. 공식 데이터 설명은 두 컬럼을 개업률·폐업률 수치로 제공하지만 상한을 명시하지 않으므로, 현재의 0~100 기준은 확정 오류가 아니라 검토 기준으로 해석한다. |
| 4.6 분기 코드 형식 | 정상 | 분기형 데이터 6종 모두 2021년 1분기(`20211`)부터 2026년 1분기(`20261`)까지 21개 분기를 포함하며 형식 오류는 0건이다. |
| 4.7 총계와 구성값 일치 | 대체로 정상 | 점포, 매출, 상주인구, 가구, 직장인구는 모두 허용 오차 안에서 일치한다. 유동인구는 2,610행이 현재 허용 오차 1을 넘지만 최대 차이가 3명에 불과하므로 집계·반올림 차이로 해석할 수 있다. 유동인구에 한해 허용 오차를 3으로 조정하는 것이 적절하다. |

### 종합 판단

핵심 키, 상권 코드 참조, 분기 형식과 주요 합계 관계는 전반적으로 정상이다. 현재 즉시 제거해야 할 명확한 오류값은 발견되지 않았다. 후속 작업에서는 상권명 2개 코드의 표기 통일 여부를 정하고, 개·폐업률 100 초과 값을 원천 산식에 따라 해석하며, 유동인구 합계 검사 허용 오차를 3으로 조정하는 것이 좋다.

참고: [서울시 상권분석서비스 점포-상권 데이터 설명](https://data.seoul.go.kr/dataList/OA-15577/S/1/datasetView.do?tab=A)

## 6. 검토 항목 추가 탐색

### 6.1 개업률·폐업률 100 초과

100을 초과한 비율의 건수와 최댓값을 확인하고, 해당 행의 전체 점포 수와 개·폐업 점포 수를 함께 살펴본다. 전체 점포 수보다 한 분기 동안 개업하거나 폐업한 점포 수가 많다면 비율은 100을 초과할 수 있다.

In [ ]:
stores = datasets["stores"]

rate_summary = pd.DataFrame({
    "항목": ["개업률", "폐업률"],
    "100 초과 행 수": [
        (stores["개업_율"] > 100).sum(),
        (stores["폐업_율"] > 100).sum(),
    ],
    "최댓값": [stores["개업_율"].max(), stores["폐업_율"].max()],
})

display(rate_summary)

rate_review = stores.loc[
    (stores["개업_율"] > 100) | (stores["폐업_율"] > 100),
    [
        "기준_년분기_코드", "상권_코드명", "서비스_업종_코드명",
        "유사업종_점포_수", "개업_점포_수", "개업_율",
        "폐업_점포_수", "폐업_율",
    ],
]

rate_review.sort_values(
    ["폐업_율", "개업_율"],
    ascending=False,
).head(30)

### 6.2 유동인구 성별 합계 차이

`총 유동인구 - 남성 유동인구 - 여성 유동인구`를 계산해 차이의 전체 분포를 확인한다. 차이가 작은 정수 범위에만 집중되어 있으면 개별 성별 값의 반올림 과정에서 발생한 차이로 볼 수 있다.

In [ ]:
floating = datasets["floating_population"]

floating_difference = (
    floating["총_유동인구_수"]
    - floating["남성_유동인구_수"]
    - floating["여성_유동인구_수"]
)

floating_difference_df = (
    floating_difference.value_counts()
    .sort_index()
    .rename_axis("합계 차이")
    .reset_index(name="행 수")
)

floating_difference_df

### 6.3 상권명 변경 이력

여러 명칭이 연결된 상권 코드 2개의 분기별 명칭을 확인한다. 특정 분기를 기준으로 모든 데이터에서 동일하게 명칭이 바뀌었다면 무작위 오류보다는 원천의 명칭 정비 또는 변경으로 해석할 수 있다.

In [ ]:
review_codes = [3110024, 3110379]
name_history_list = []

for name, df in datasets.items():
    if "기준_년분기_코드" not in df.columns:
        continue

    history = df.loc[
        df["상권_코드"].isin(review_codes),
        ["기준_년분기_코드", "상권_코드", "상권_코드명"],
    ].drop_duplicates()

    history.insert(0, "데이터", name)
    name_history_list.append(history)

name_history_df = pd.concat(name_history_list, ignore_index=True)

name_history_df.sort_values(
    ["상권_코드", "기준_년분기_코드", "데이터"]
)

### 추가 탐색 결과

- **개·폐업률:** 개업률 100 초과는 14행, 폐업률 100 초과는 767행이다. 대표적으로 전체 점포가 1개인데 한 분기에 2개가 개업하거나 2~3개가 폐업한 경우 비율이 200~300으로 나타난다. 최댓값은 개업률 200, 폐업률 500이다. 따라서 100 초과를 곧바로 오류로 처리하지 않고, 음수 여부만 최소 오류 조건으로 두는 것이 적절하다.
- **유동인구 합계:** 전체 34,633행의 차이가 모두 -3~3 범위에 있다. 0인 행이 17,167개로 가장 많으며 최대 절대 차이가 3명에 불과하므로 반올림 차이로 판단한다. 합계 검사의 허용 오차를 3으로 변경할 수 있다.
- **상권명:** 두 상권 모두 2024년 4분기까지 과거 명칭을 사용하고 2025년 1분기부터 새 명칭으로 일괄 변경됐다. 무작위 오탈자보다는 원천의 명칭 변경 이력에 가깝다. 시계열 분석의 결합 키는 상권 코드를 사용하고, 화면 표시용 명칭은 최신 `trade_area` 명칭을 사용하는 것이 적절하다.